In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Reshape, Flatten, LeakyReLU, BatchNormalization, Dropout, Conv2D, Conv2DTranspose
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
import copy

In [ ]:
df_org = pd.read_csv('/kaggle/input/emo-map-challenge/train_dataset.csv')
tdf_org = pd.read_csv('/kaggle/input/emo-map-challenge/test_dataset.csv')
df = copy.deepcopy(df_org)
tdf = copy.deepcopy(tdf_org)

In [ ]:
df

In [ ]:
disgust_images = df[df['emotion'] == 1]['pixels']
img_rows, img_cols, channels = 48, 48, 1
img_shape = (img_rows, img_cols, channels)
latent_dim = 100

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

In [ ]:
def preprocess_images(image_list):
    images = np.array([np.fromstring(pixels, dtype=int, sep=' ').reshape(48, 48) for pixels in image_list])
    images = np.expand_dims(images, axis=-1)
    images = (images - 127.5) / 127.5
    return images

In [ ]:
def build_generator():
    model = Sequential()
    model.add(Dense(256, input_dim=latent_dim))
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Dense(512))
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Dense(1024))
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Dense(np.prod(img_shape), activation='tanh'))
    model.add(Reshape(img_shape))
    return model

def build_discriminator():
    model = Sequential()
    model.add(Conv2D(32, kernel_size=3, strides=2, input_shape=img_shape, padding="same"))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dropout(0.25))
    model.add(Conv2D(64, kernel_size=3, strides=2, padding="same"))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dropout(0.25))
    model.add(Conv2D(128, kernel_size=3, strides=2, padding="same"))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dropout(0.25))
    model.add(Conv2D(256, kernel_size=3, strides=1, padding="same"))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dropout(0.25))
    model.add(Flatten())
    model.add(Dense(1, activation='sigmoid'))
    return model

def build_gan(generator, discriminator):
    discriminator.trainable = True
    model = tf.keras.Sequential()
    model.add(generator)
    model.add(discriminator)
    return model

generator = build_generator()
discriminator = build_discriminator()
gan = build_gan(generator, discriminator)

optimizer = Adam(0.0002, 0.5)
discriminator.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
gan.compile(loss='binary_crossentropy', optimizer=optimizer)

In [ ]:
print("Discriminator Summary:")
discriminator.summary()

print("\nGenerator Summary:")
generator.summary()

print("\nGAN Summary:")
gan.summary()

In [ ]:
def train_gan(epochs, batch_size=128, save_interval=200):
    half_batch = batch_size // 2
    augmented_images = preprocess_images(disgust_images)

    for epoch in range(epochs):
        noise = np.random.normal(0, 1, (half_batch, latent_dim))
        gen_images = generator.predict(noise)

        real_images = augmented_images[np.random.randint(0, augmented_images.shape[0], half_batch)]

        combined_images = np.concatenate([gen_images, real_images])
        labels = np.concatenate([np.zeros((half_batch, 1)), np.ones((half_batch, 1))])

        labels += 0.05 * np.random.random(labels.shape)
    
        d_loss = discriminator.train_on_batch(combined_images, labels)

        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        misleading_labels = np.ones((batch_size, 1))
        g_loss = gan.train_on_batch(noise, misleading_labels)


        if epoch % save_interval == 0:
            print(f"{epoch} [D loss: {d_loss[0]} | Acc: {d_loss[1] * 100}] [G loss: {g_loss}]")
            save_generated_images(epoch)


def save_generated_images(epoch, examples=9, dim=(3, 3), figsize=(10, 10)):
    noise = np.random.normal(0, 1, (examples, latent_dim))
    gen_images = generator.predict(noise)
    gen_images = 0.5 * gen_images + 0.5  

    plt.figure(figsize=figsize)
    for i in range(examples):
        plt.subplot(dim[0], dim[1], i + 1)
        plt.imshow(gen_images[i].reshape(48, 48), cmap='gray')
        plt.axis('off')
    plt.tight_layout()
    plt.savefig(f'gan_generated_image_epoch_{epoch}.png')
    plt.show()

In [ ]:
def show_augmented_images():
    augmented_images = preprocess_images(disgust_images)
    augmented_images = augmented_images[:9]
    augmented_images = augmented_images.reshape(augmented_images.shape[0], 48, 48, 1)

    i = 0
    for batch in datagen.flow(augmented_images, batch_size=9):
        plt.figure(figsize=(10, 10))
        for i in range(9):
            plt.subplot(3, 3, i + 1)
            plt.imshow(batch[i].reshape(48, 48), cmap='gray')
            plt.axis('off')
        plt.show()
        break

show_augmented_images()

In [ ]:
train_gan(epochs=5000, batch_size=32, save_interval=100)